# To publish or not to publish

**CYPLAN 101 · Public data, public trust**

A state department runs a benefits program. It has a table of how many people are enrolled in each California county, broken out by race/ethnicity and age group.

- A journalist has filed a **Public Records Act** request for the table.
- An advocacy group wants it posted on **data.ca.gov**.
- Some cells count only a handful of people.

In the next 30 minutes you'll explore the table, protect it, try to break that protection, and then recommend what the department should do.

> **All data here is synthetic.** County names are real and county sizes are roughly realistic, but every count and every person is made up. No real program, community, or tribe is represented.

**How this works:** run each cell in order with **Shift + Enter**. Cells marked ✏️ ask you to fill in a blank, shown as `...`. Hints are in the comments. You can work alone or with a partner.

**If you see an error:**
- An error that mentions `ellipsis` or "Replace the ..." means a ✏️ blank still needs filling in.
- `NameError: name '...' is not defined` usually means an earlier ✏️ cell wasn't finished or run. Go back, fix it, and run it again.

In [ ]:
# Run this cell first. It loads the data and some helper functions.
import pandas as pd
import helpers as h

h.set_style()
enrollment = h.load_enrollment()
print(f"Loaded {len(enrollment):,} rows")

---
## Part 1 · Explore the table (5 min)

Each row is one **cell** of the published table: one county, one race/ethnicity group, one age group, and the number of people enrolled.

In [ ]:
# Look at the first few rows
enrollment.head()

✏️ **1a.** Which cells have the smallest counts? Show the 10 smallest **non-zero** counts.

In [ ]:
# Hint: keep rows where enrolled > 0, then use .nsmallest(10, "enrolled")
nonzero = enrollment[enrollment["enrolled"] > 0]
smallest = ...
smallest

✏️ **1b.** How many cells count between 1 and 10 people?

In [ ]:
# Hint: combine two conditions with &, and wrap each one in parentheses.
small_cells = enrollment[(enrollment["enrolled"] > 0) & (enrollment["enrolled"] < ...)]
len(small_cells)

**Look at the result:** which counties and groups show up most often among the small cells? (No code needed. Just look at `small_cells` or `smallest`.)

---
## Part 2 · Suppress the small cells (7 min)

California's health and human services agencies use shared **de-identification guidelines**. A common rule: hide (suppress) any count based on **fewer than 11 people**. Zeros are shown.

The helper `h.suppress()` applies that rule and adds two columns: `published` (what the public would see) and `suppressed` (True if hidden).

✏️ **2a.** Suppress the table using a threshold of 11.

In [ ]:
published = h.suppress(enrollment, threshold=...)
published[published["suppressed"]].head(10)

**2b.** Who disappears? The table below shows, for each group, the share of cells hidden and the share of **people** in hidden cells. Run it for the whole state first.

In [ ]:
h.suppression_by_group(published)

✏️ **2c.** Statewide, the numbers look small. Now look only at **small counties** (under 50,000 people).

In [ ]:
# Hint: keep rows where county_population is less than 50_000
small_counties = published[published["county_population"] < ...]
summary = h.suppression_by_group(small_counties)
h.plot_hidden_by_group(summary, title="Small counties: share of enrollees hidden")

✏️ **Your answer (1–2 sentences):** In small counties, which groups lose the most visibility? What might that mean for a community trying to use this data to make the case for resources?

*Your answer here.*

**Try it (optional):** go back to 2a and change the threshold to 5 or 20. How do the results change? The threshold is a choice, not a law of nature.

---
## Part 3 · Work backward from the totals (5 min)

The department also publishes a **total for each county and race/ethnicity group** (all ages combined). That seems harmless. But if only one cell in a row is hidden, anyone can subtract the visible cells from the total.

In [ ]:
totals = h.published_totals(enrollment)
totals.head()

✏️ **3a.** Use the helper to find hidden cells that can be recovered from the totals. How many are there?

In [ ]:
# Hint: h.recover_hidden_cells(published_table, totals)
recovered = h.recover_hidden_cells(..., totals)
print(f"{len(recovered)} hidden cells can be recovered")
recovered.head(10)

Compare `recovered_value` and `true_value`. Suppression alone didn't protect these cells.

The fix is **complementary suppression**: hide extra cells in the same row so the hidden ones can't be worked out. The helper `h.complementary_suppress()` does this.

✏️ **3b.** Apply complementary suppression with a threshold of 11, then check how many cells can still be recovered.

In [ ]:
protected = h.complementary_suppress(enrollment, threshold=...)
still_recoverable = h.recover_hidden_cells(protected, totals)
print(f"Cells hidden now: {protected['suppressed'].sum()}")
print(f"Cells still recoverable: {len(still_recoverable)}")

In [ ]:
# What did that cost? Same chart as Part 2, after complementary suppression.
summary_protected = h.suppression_by_group(protected[protected["county_population"] < 50_000])
h.plot_hidden_by_group(summary_protected, title="Small counties, after complementary suppression")

**Look at the result:** privacy went up. Who paid for it?

---
## Part 4 · Link two datasets (7 min)

The department also shared a **record-level extract** with a researcher. Names were removed, so it's labeled "de-identified." It covers a few small rural ZIP codes.

Separately, there's a **public roster** of residents in the same ZIP codes, with names, ZIP code, birth year, and sex. (Both are fictional.)

In [ ]:
extract = h.load_extract()
roster = h.load_roster()
extract.head()

In [ ]:
roster.head()

✏️ **4a.** How unique are people in the extract? Find the percent of records whose ZIP code, birth year, and sex appear only once.

In [ ]:
# Hint: pass a list of the three column names
h.share_unique(extract, ["zip", ..., ...])

✏️ **4b.** Join the extract to the roster on those same three columns. Each match links a "de-identified" record to a name.

In [ ]:
# Hint: use the same list of columns as in 4a
matches = extract.merge(roster, on=[...])
matches.head()

A record that matches **exactly one** person in the roster has been re-identified.

In [ ]:
matches_per_record = matches.groupby("record_id")["name"].transform("size")
reidentified = matches[matches_per_record == 1]
print(f"{reidentified['record_id'].nunique()} of {len(extract)} 'de-identified' records now have a name")
reidentified[["record_id", "name", "zip", "birth_year", "sex", "monthly_benefit"]].head(10)

✏️ **4c.** A local reporter knows that **Ruth Calloway** (fictional) lives in ZIP code 96120 and was born in 1957. Can you find her benefit amount?

In [ ]:
# Hint: filter reidentified where the name column equals "Ruth Calloway"
reidentified[reidentified["name"] == ...]

No names, Social Security numbers, or addresses were in the extract. Three ordinary details were enough.

---
## Part 5 · Decide (6 min)

You're advising the department's director. Remember there are **two decisions**:

1. How to respond to the **Public Records Act request**, which the department is legally required to answer.
2. Whether and how to post the table on **data.ca.gov**, which is discretionary.

✏️ **Write a one-paragraph recommendation.** Address:

- **Privacy:** what you'd suppress or aggregate, and why (think about Part 3)
- **Transparency:** what the public, and the journalist, will still be able to see
- **Who's missing:** which communities lose visibility, and what you'd do about it
- **Tribal consultation:** who you'd talk to before posting, and when

*Your recommendation here.*